# 10. Futility-анализ


## План занятия

1. **Постановка задачи** — efficacy vs futility.
2. **Аппарат** — стандартизованная статистика $Z_t$, информация, дрейф, $B(t)$.
3. **Conditional Power (CP)** — шанс дожить до значимости.
4. **Predictive Power (PPoS)** — CP, усреднённый по неопределённости эффекта.
5. **β-spending** — заранее фиксированная futility-граница (зеркало α-spending).
6. **Binding vs non-binding** — как futility влияет на ошибку I рода.
7. **Влияние на ошибки I/II рода** — Monte-Carlo и операционные характеристики.
8. **Практические рекомендации.**
9. **Сквозной пример: сравнение всех методов** — $z$-тест, GST, GST + futility (binding / non-binding / CP / PPoS).
10. **Ненормальность метрики** и **итоги**.


## Библиотеки

In [45]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

pio.templates.default = "plotly_white"

SEED = 12345
rng = np.random.default_rng(SEED)

# Единая палитра исходов на всю лекцию
C_FUT = "#d95f02"   # оранжевый — futility
C_EFF = "#1b9e77"   # зелёный — efficacy
C_CONT = "#999999"  # серый — продолжаем


def style(fig, height=440, title=None, xlabel=None, ylabel=None):
    """Единый стиль всех графиков: сетка, легенда, подписи."""
    fig.update_layout(
        template="plotly_white",
        height=height,
        title=dict(text=title, font=dict(size=15)),
        xaxis_title=xlabel, yaxis_title=ylabel,
        margin=dict(l=60, r=20, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, font=dict(size=11)),
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=True)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=True)
    return fig

In [35]:
@dataclass(frozen=True)
class Design:
    """Параметры теста непрерывной метрики (средний чек)."""
    alpha: float = 0.05        # односторонняя ошибка I рода
    beta: float = 0.20         # ошибка II рода (power = 0.80)
    sigma: float = 100.0       # СКО метрики, руб.
    mde: float = 10.0          # абсолютный MDE, руб.
    K: int = 5
    t: tuple = (0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0)  # доли информации (моменты взглядов)

    @property
    def delta(self) -> float:
        return self.mde / self.sigma

    @property
    def theta_mde(self) -> float:
        return norm.ppf(1 - self.alpha) + norm.ppf(1 - self.beta)

    @property
    def i_max(self) -> float:
        return (self.theta_mde / self.delta) ** 2

    @property
    def n_max(self) -> float:
        return 2 * self.sigma ** 2 * self.i_max

design = Design()
design

Design(alpha=0.05, beta=0.2, sigma=100.0, mde=10.0, K=5, t=(0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0))

## 0. Постановка задачи


Есть две разновидности ранней остановки эксперимента:

- **Остановка по эффективности (efficacy)** — эффект статистически доказан до оконания эксперимента. Останавливаемся, потому что *победили*.
- **Остановка по бесперспективности (futility)** — эффекта, скорее всего, нет или он меньше MDE. Останавливаемся, чтобы не жечь трафик на заведомо «мёртвый» тест.

**Остановка по бесперспективности (futility)** появляется ка кестественный ответ на следующий вопрос:
> *Если на промежуточном взгляде эффект слабее MDE (или его нет вовсе), с какой вероятностью тест всё-таки дойдёт до значимости к концу?*

Если эта вероятность мала — тратить оставшийся трафик бессмысленно, и можно завершить тест.

## 1. Аппарат: стандартизованная статистика и информация

<details>
<summary>1.1. Вывод распределения $Z_t$</summary>

Данные: контроль $A$ и тест $B$, по $n_t$ наблюдений каждая, независимые, из нормальных распределений с общей дисперсией $\sigma^2$. Оценка эффекта — разность средних: $\hat\delta_t=\bar X_B-\bar X_A$.

**Дисперсия среднего.** $\mathrm{Var}(\bar X_A)=\mathrm{Var}\!\left(\frac1{n_t}\sum X_{A,i}\right)=\frac{1}{n_t^2}\sum\mathrm{Var}(X_{A,i})=\frac{1}{n_t^2}\cdot n_t\sigma^2=\frac{\sigma^2}{n_t}$. Аналогично для $B$. Так как выборки независимы, дисперсии складываются: $\mathrm{Var}(\hat\delta_t)=2\sigma^2/n_t$, то есть $SE_t=\sigma\sqrt{2/n_t}$.

**Стандартизация.** $Z_t=\hat\delta_t/SE_t$. Тогда $\mathbb E[Z_t]=\delta/SE_t$, $\mathrm{Var}(Z_t)=1$, то есть $Z_t\sim\mathcal N\big(\delta/SE_t,\,1\big)$.

**Информация.** $I_t=1/SE_t^2=n_t/(2\sigma^2)$ — обратная дисперсия оценки. Вводим $t=I_t/I_{max}\approx n_t/n_{max}$ и дрейф $\theta=\delta\sqrt{I_{max}}$. Тогда $\delta\sqrt{I_t}=\delta\sqrt{I_{max}}\cdot\sqrt{I_t/I_{max}}=\theta\sqrt t$, и

$$\boxed{Z_t\sim\mathcal N(\theta\sqrt t,\,1)}$$

</details>

<details>
<summary>1.2. Откуда $\theta_{MDE}=z_{1-\alpha}+z_{1-\beta}$</summary>

Это решение уравнения на мощность. При $t=1$: $Z_1\sim\mathcal N(\theta,1)$. Мощность $=P(Z_1>z_{1-\alpha})=1-\Phi(z_{1-\alpha}-\theta)$. Требуем $=1-\beta$: $1-\Phi(z_{1-\alpha}-\theta)=1-\beta\iff\Phi(z_{1-\alpha}-\theta)=\beta\iff z_{1-\alpha}-\theta=-z_{1-\beta}$. Отсюда

$$\boxed{\theta_{MDE}=z_{1-\alpha}+z_{1-\beta}}$$

</details>

<details>
<summary>1.3. Независимые приращения и ковариация</summary>

**Факт 1.** Среднее по $n_t$ наблюдениям раскладывается на старый ($n_s$) и новый ($n_t-n_s$) куски: $\bar X_B^{(t)}=\frac{n_s}{n_t}\bar X_B^{(s)}+\frac{n_t-n_s}{n_t}\bar X_B^{(new)}$. Вычитая для $A$ и $B$ и деля на $SE_t$, получаем $Z_t=\sqrt{\frac{s}{t}}Z_s+\sqrt{\frac{t-s}{t}}Z_{new}$.

**Факт 2.** Каждый кусок пересчитывается в свою нормированную статистику: вес старого куска $=\frac{n_s}{n_t}\sqrt{\frac{n_t}{n_s}}=\sqrt{s/t}$, нового — $\sqrt{(t-s)/t}$. Итог:

$$\boxed{Z_t=\sqrt{\frac{s}{t}}\,Z_s+\sqrt{\frac{t-s}{t}}\,Z_{new}}$$

где $Z_{new}\perp Z_s$, $\mathrm{Var}(Z_{new})=1$. Проверка нормировки: $s/t+(t-s)/t=1$. ✓

**Ковариация.** $\mathrm{Cov}(Z_s,Z_t)=\sqrt{\frac{s}{t}}\,\mathrm{Cov}(Z_s,Z_s)+0=\sqrt{s/t}$. Так как дисперсии равны 1, это и корреляция: $\mathrm{corr}(Z_s,Z_t)=\sqrt{s/t}$, то есть $R^2=s/t$ — доля общей информации.

</details>

<details>
<summary>1.4. Переход к $B(t)$ и его физический смысл</summary>

В $Z$-единицах приращение $Z_t-Z_s$ имеет дисперсию $2-2\sqrt{s/t}$, зависящую от обоих моментов. Умножаем на $\sqrt t$: $B(t)=\sqrt t\,Z_t$. Тогда

$$\mathbb E[B(t)]=\theta t,\quad \mathrm{Var}[B(t)]=t,\quad \mathrm{Cov}(B(s),B(t))=s=\min(s,t),$$

и $\mathrm{Cov}(B(s),B(t)-B(s))=s-s=0$. Так как $(B(s),B(t))$ совместно нормальны, нулевая ковариация = независимость:

$$B(t)-B(s)\perp B(s),\qquad B(t)-B(s)\sim\mathcal N(\theta(t-s),\,t-s).$$

**Физический смысл.** Подставив $Z_t=\hat\delta_t\sqrt{I_t}$: $B(t)=\frac{1}{2\sigma^2\sqrt{I_{max}}}\sum_{i=1}^{n_t}\Delta_i$, где $\Delta_i=X_{B,i}-X_{A,i}$. $B$ — с точностью до константы **накопленная сумма по пользователям**. Поэтому независимость приращений — точное свойство дизайна, а не приближение. $B$ — броуновское движение с дрейфом $\theta$.

**Переход границ:** $Z_k>c_k\iff B(t_k)>c_k\sqrt{t_k}=:b_k$.

</details>


In [3]:
def z_stat(xA: np.ndarray, xB: np.ndarray, sigma: float) -> float:
    """Двухвыборочный z-тест непрерывной метрики (известная дисперсия)."""
    nA, nB = xA.size, xB.size
    se = sigma * np.sqrt(1 / nA + 1 / nB)
    return (xB.mean() - xA.mean()) / se


def simulate_b(rng, theta: float, t: np.ndarray, n_sim: int = 200) -> np.ndarray:
    """Траектории B(t) = sqrt(t)*Z_t — броуновское движение с дрейфом theta."""
    dt = np.diff(np.concatenate(([0.0], t)))
    inc = rng.normal(loc=theta * dt, scale=np.sqrt(dt), size=(n_sim, len(t)))
    return np.cumsum(inc, axis=1)


In [44]:
t = np.asarray(design.t)

modes = [
    (0.0,              "H<sub>0</sub> (θ = 0)",    C_CONT),
    (design.theta_mde, "H<sub>1</sub> (θ = MDE)",  C_EFF),
]

B_by_mode = [(theta, name, color, simulate_b(rng, theta, t, 250))
             for theta, name, color in modes]

x_ticks = np.linspace(t.min(), t.max(), 20)
y_all   = np.concatenate([B.ravel() for *_, B in B_by_mode])
y_lo    = min(y_all.min(), 0.0)
y_hi    = max(y_all.max(), modes[1][0] * t.max())
y_ticks = np.linspace(y_lo, y_hi, 20)

# Подписи панелей — с расшифровкой смысла, чтобы легенда была не нужна
subplot_titles = (
    "H<sub>0</sub> (θ = 0)",
    "H<sub>1</sub> (θ = MDE)",
)

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    horizontal_spacing=0.06,
    subplot_titles=subplot_titles,
)

for col, (theta, name, color, B) in enumerate(B_by_mode, start=1):
    for i in range(B.shape[0]):
        fig.add_trace(
            go.Scatter(
                x=t, y=B[i], mode="lines",
                line=dict(color=color, width=0.8),
                opacity=0.25,
                showlegend=False,
                hoverinfo="skip",
            ),
            row=1, col=col,
        )
    # Чёрная штриховая линия дрейфа (легенды нет — подпишем аннотацией ниже)
    fig.add_trace(
        go.Scatter(
            x=t, y=theta * t, mode="lines",
            line=dict(color="black", width=2.5, dash="dash"),
            showlegend=False,
            name=f"E[B(t)] = {theta:.2f}·t",
        ),
        row=1, col=col,
    )

    # Небольшая подпись у линии дрейфа — вместо легенды
    fig.add_annotation(
        text=f"E[B(t)] = {theta:.2f}·t",
        x=t.max(), y=theta * t.max(),
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=11, color="black"),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="rgba(0,0,0,0.35)", borderwidth=1,
        row=1, col=col,
    )

fig.update_xaxes(
    tickmode="array", tickvals=x_ticks, tickformat=".2f",
    title_text="Доля информации t", automargin=True,
)
fig.update_yaxes(
    tickmode="array", tickvals=y_ticks, tickformat=".2f",
    automargin=True,
)
fig.update_yaxes(title_text="B(t)", col=1)

fig.update_layout(
    autosize=True,
    height=720,
    margin=dict(t=70, b=80, l=70, r=40),
    title=dict(
        text="Броуновское движение при H<sub>0</sub> и H<sub>1</sub>",
        x=0.5, xanchor="center",
        y=0.98, yanchor="top", yref="container",
        font=dict(size=16),
    ),
    showlegend=False,
)

fig

## 2. Conditional Power (CP): условная вероятность «добить» тест

<details>
<summary>2.1. Определение и полный вывод формулы (развернуть)</summary>

**Определение.** $\mathrm{CP}(\theta)=P_\theta(Z_1>z_{1-\alpha}\mid Z_t=z_t)$ — вероятность дойти до значимости при заданном дрейфе $\theta$ и текущих данных. Это и есть ответ на вопрос «стоит ли тратить оставшийся трафик».

**Вывод.**

1. $Z_1=B(1)=B(t)+\big(B(1)-B(t)\big)=\sqrt t\,z_t+\big(B(1)-B(t)\big)$.
2. Приращение независимо и $\sim\mathcal N(\theta(1-t),\,1-t)$, поэтому условно $Z_1\mid z_t\sim\mathcal N(\sqrt t\,z_t+\theta(1-t),\;1-t)$.
3. Вероятность попасть в $\{Z_1>z_{1-\alpha}\}$ — хвост этого распределения:

$$\mathrm{CP}(\theta)=1-\Phi\!\left(\frac{z_{1-\alpha}-\sqrt t\,z_t-\theta(1-t)}{\sqrt{1-t}}\right).$$

**Монотонность по $z_t$** (нужна для инверсии в разделе 3): $\frac{d\,\mathrm{CP}}{dz_t}=\varphi(u)\sqrt{\frac{t}{1-t}}>0$, поэтому уравнение $\mathrm{CP}=\gamma$ имеет единственный корень.

**Предельные случаи.** При $t\to0$: $\mathrm{CP}\to1-\Phi(z_{1-\alpha}-\theta)$ — априорная мощность дизайна. При $t\to1$: ступенька в точке $z_t=z_{1-\alpha}$ — судьба решена.

</details>

**Выбор $\theta$ — самый тонкий момент.**

- $\theta = 0$ (под $H_0$): консервативно, редко останавливает.
- $\theta = \hat\theta_t = z_t/\sqrt t$ (тренд): оптимистично — ранняя оценка шумная.
- $\theta = \theta_{MDE}$ (дизайновый эффект): стандартный ориентир.

Правило futility: **если $\mathrm{CP} < \gamma$ — останавливаемся** (типично $\gamma \in [0.10, 0.20]$).


In [5]:
def conditional_power(z_t, t, theta, z_alpha=1.6448536):
    """CP: вероятность Z_1 > z_alpha при наблюдаемом z_t и истинном дрейфе theta."""
    z_t = np.asarray(z_t, dtype=float)
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - theta * (1 - t)) / np.sqrt(1 - t))


def ppos(z_t, t, theta0, tau, z_alpha=1.6448536):
    """Predictive probability of success (нормальный prior, замкнутая форма)."""
    z_t = np.asarray(z_t, dtype=float)
    sigma2 = 1.0 / (1.0 / tau**2 + t)
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - mu * (1 - t))
                          / np.sqrt((1 - t) + (1 - t)**2 * sigma2))


In [6]:
z_alpha = norm.ppf(1 - design.alpha)
z_grid = np.linspace(-3, 3, 300)
tt = 0.5

fig = go.Figure()
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, 0.0, z_alpha),
                         mode="lines", line=dict(color=C_CONT, width=2.5), name="CP под H0 (θ = 0)"))
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, design.theta_mde, z_alpha),
                         mode="lines", line=dict(color=C_EFF, width=2.5), name="CP под MDE (θ = θ_MDE)"))
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, z_grid / np.sqrt(tt), z_alpha),
                         mode="lines", line=dict(color=C_FUT, width=2.5), name="CP под трендом (оптимистично)"))
fig.add_hline(y=0.15, line_dash="dash", line_color="black", annotation_text="порог γ = 0.15")
fig = style(fig, title=f"CP при t = {tt}: выбор θ кардинально меняет ответ",
            xlabel="наблюдаемая статистика z_t", ylabel="conditional power")
fig


## 2.2. Карта CP: интерактивная

Карта CP на плоскости $(t, z)$ показывает, почему время взгляда так важно: на ранних взглядах карта почти плоская (судьбу решает будущее), к концу «заостряется» в ступеньку. Красный контур $\mathrm{CP}=\gamma$ — граница «стоп / продолжаем». Двигайте $\theta$ и $\gamma$ — меняется вся карта и положение границы.


In [7]:
def plot_cp_heatmap(theta=design.theta_mde, gamma=0.15):
    t_grid = np.linspace(0.05, 0.95, 120)
    z_grid = np.linspace(-3, 3, 240)
    T, Z = np.meshgrid(t_grid, z_grid)
    CP = conditional_power(Z, T, theta, z_alpha)
    a_cp = np.array([cp_to_boundary(t_, gamma, theta, z_alpha) for t_ in t_grid])

    fig = go.Figure()
    fig.add_trace(go.Heatmap(z=CP, x=t_grid, y=z_grid, colorscale="Viridis",
                             colorbar=dict(title="CP"), showscale=True))
    fig.add_trace(go.Scatter(x=t_grid, y=a_cp, mode="lines", line=dict(color="red", width=3),
                             name=f"граница CP = {gamma}"))
    fig = style(fig, height=520, title="Conditional power под MDE: контур — граница futility-остановки",
                xlabel="доля информации t", ylabel="наблюдаемая статистика z_t")
    fig.update_layout(coloraxis=dict(colorscale="Viridis"))
    return go.FigureWidget(fig)

# cp_to_boundary объявлена ниже; для запуска этого интерактива она нужна — определим её здесь заранее
def cp_to_boundary(t, gamma, theta, z_alpha=1.6448536):
    f = lambda z: conditional_power(z, t, theta, z_alpha) - gamma
    return brentq(f, -8, 8)

interact(plot_cp_heatmap,
         theta=FloatSlider(value=design.theta_mde, min=0, max=4, step=0.1, description="θ"),
         gamma=FloatSlider(value=0.15, min=0.05, max=0.40, step=0.05, description="γ"))


interactive(children=(FloatSlider(value=2.4864748605243863, description='θ', max=4.0), FloatSlider(value=0.15,…

<function __main__.plot_cp_heatmap(theta=np.float64(2.4864748605243863), gamma=0.15)>

## 3. Перевод порога CP в границу (инверсия)

<details>
<summary>Развернуть</summary>

Порог $\mathrm{CP}<\gamma$ перепишем в терминах статистики. Определим $a_{CP}$ как решение $\mathrm{CP}(a_{CP};t,\theta)=\gamma$. Так как CP монотонно растёт по $z_t$, это уравнение имеет единственный корень, и $\mathrm{CP}(z_t)<\gamma\iff z_t<a_{CP}(t)$.

**Различие типов границ.** CP-граница строится из условного расчёта на каждом взгляде и **не гарантирует заранее** суммарную ошибку II рода. Альтернатива — зафиксировать границу *до* данных с гарантией контроля: β-spending (раздел 5).

</details>


In [8]:
def cp_to_boundary(t, gamma, theta, z_alpha=1.6448536):
    """z_t, при котором CP(z_t; t, theta) == gamma."""
    f = lambda z: conditional_power(z, t, theta, z_alpha) - gamma
    return brentq(f, -8, 8)


In [9]:
tt_grid = np.linspace(0.05, 0.95, 200)
fig = go.Figure()
for theta, name, color in [(0.0, "по CP под H0", C_CONT), (design.theta_mde, "по CP под MDE", C_FUT)]:
    a = [cp_to_boundary(t_, 0.15, theta, z_alpha) for t_ in tt_grid]
    fig.add_trace(go.Scatter(x=tt_grid, y=a, mode="lines", line=dict(color=color, width=2.5), name=f"граница {name}"))
fig.add_trace(go.Scatter(x=[0.2, 0.5, 0.8, 0.95], y=[1.6, 1.1, 0.55, 0.3], mode="lines+markers",
                         line=dict(color="black", width=1.8), marker=dict(size=6),
                         name="пример наблюдаемой статистики"))
fig = style(fig, title="Futility-граница из порога CP = 0.15 (данные-зависимая)",
            xlabel="доля информации t", ylabel="граница a(t) на шкале z")
fig


## 4. Predictive Power (PPoS): усредняем CP по неопределённости эффекта

<details>
<summary>4.1. Идея и вывод апостериора (развернуть)</summary>

CP фиксирует $\theta$. Но подставляя «текущий тренд», мы притворяемся, что шумная оценка — истина. PPoS усредняет CP по апостериорной неопределённости: $\mathrm{PPoS}=\int \mathrm{CP}(\theta)\,\pi(\theta\mid D_t)\,d\theta$.

Априор $\theta\sim\mathcal N(\theta_0,\tau^2)$, правдоподобие $Z_t\mid\theta\sim\mathcal N(\theta\sqrt t,1)$ эквивалентно $y=z_t/\sqrt t\sim\mathcal N(\theta,1/t)$. Апостериор нормален, точности складываются:

$$\frac1{\sigma_p^2}=\frac1{\tau^2}+t,\qquad \mu_p=\sigma_p^2\left(\frac{\theta_0}{\tau^2}+\sqrt t\,z_t\right).$$

</details>

<details>
<summary>4.2. Замкнутая форма PPoS (развернуть)</summary>

Числитель CP линеен по $\theta$: с $\theta=\mu_p+W$, $W\sim\mathcal N(0,\sigma_p^2)$, имеем $z_{1-\alpha}-\sqrt t z_t-\theta(1-t)=A-(1-t)W$. Используем тождество $\mathbb E_W[\Phi((a-W)/b)]=\Phi(a/\sqrt{b^2+\sigma_W^2})$:

$$\mathrm{PPoS}=1-\Phi\!\left(\frac{z_{1-\alpha}-\sqrt t\,z_t-\mu_p(1-t)}{\sqrt{(1-t)+(1-t)^2\sigma_p^2}}\right).$$

Знаменатель больше, чем у CP, на слагаемое за неопределённость эффекта — поэтому PPoS ниже CP-под-трендом. CP — частный случай PPoS с $\sigma_p^2\to0$.

</details>

**Общая рамка.** CP и PPoS — две инкарнации **stochastic curtailment**: останавливаем тест, когда условная вероятность полезного исхода пренебрежимо мала. PPoS удобнее для бизнеса — это «вероятность успеха в процентах». График ниже интерактивный.


In [10]:
def plot_ppos(z_t=0.4, t=0.5, theta0=0.0, tau=0.8):
    sigma2 = 1.0 / (1.0 / tau**2 + t)
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)
    theta_grid = np.linspace(-0.5, 4, 400)
    post = norm.pdf(theta_grid, loc=mu, scale=np.sqrt(sigma2))
    cp_theta = conditional_power(z_t, t, theta_grid, z_alpha)
    pp = ppos(z_t, t, theta0, tau, z_alpha)
    cp_trend = conditional_power(z_t, t, z_t / np.sqrt(t), z_alpha)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=theta_grid, y=post, mode="lines", line=dict(color=C_CONT, width=2.5),
                             fill="tozeroy", fillcolor="rgba(153,153,153,0.15)", name="апостериор π(θ|данные)"))
    fig.add_vline(x=mu, line_dash="dash", line_color="black", annotation_text="среднее")
    fig.add_vline(x=design.theta_mde, line_dash="dot", line_color=C_EFF, annotation_text="θ_MDE")
    fig.add_trace(go.Scatter(x=theta_grid, y=cp_theta, mode="lines", line=dict(color=C_EFF, width=2.5), name="CP(θ)"))
    fig.add_trace(go.Scatter(x=theta_grid, y=cp_theta * post, mode="lines", line=dict(color=C_FUT, width=2),
                             fill="tozeroy", fillcolor="rgba(217,95,2,0.25)", name="CP(θ)·π(θ)"))
    fig.add_hline(y=pp, line_dash="dash", line_color=C_FUT, annotation_text=f"PPoS = {pp:.3f}")
    fig.add_hline(y=cp_trend, line_dash="dash", line_color=C_CONT, annotation_text=f"CP(тренд) = {cp_trend:.3f}")
    fig = style(fig, height=480, title="PPoS = среднее CP(θ) по апостериору (ниже CP(тренд))",
                xlabel="θ", ylabel="плотность / вероятность")
    return go.FigureWidget(fig)

interact(plot_ppos,
         z_t=FloatSlider(value=0.4, min=-3, max=3, step=0.05, description="z_t"),
         t=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="t"),
         theta0=FloatSlider(value=0.0, min=-1, max=3, step=0.1, description="prior θ0"),
         tau=FloatSlider(value=0.8, min=0.05, max=3, step=0.05, description="prior τ"))


interactive(children=(FloatSlider(value=0.4, description='z_t', max=3.0, min=-3.0, step=0.05), FloatSlider(val…

<function __main__.plot_ppos(z_t=0.4, t=0.5, theta0=0.0, tau=0.8)>

## 5. β-spending: заранее фиксированная futility-граница

<details>
<summary>Логика симметрии и рекурсивный алгоритм (развернуть)</summary>

α-spending распределяет бюджет ошибки I рода по взглядам под $H_0$; β-spending — то же для ошибки II рода под $H_1$: «распределить β так, чтобы суммарная вероятность *зря остановить тест при реальном эффекте* была не больше β». Futility-границу считают под $H_1$ ($\theta=\theta_{MDE}$), efficacy — под $H_0$.

Спендинг-функции неубывающие: $\alpha(0)=\beta(0)=0$, $\alpha(1)=\alpha$, $\beta(1)=\beta$. Степенное семейство $\beta(t)=\beta t^{\rho}$: $\rho=3$ — OBF-подобная, $\rho=1$ — Pocock-подобная.

**Рекурсия на шкале $B$.** Пусть $p_k(x)$ — субплотность $B(t_k)$ среди траекторий, не остановившихся к взгляду $k-1$. Инициализация: $p_1^{eff}=\varphi(x;0,t_1)$ под $H_0$, $p_1^{fut}=\varphi(x;\theta t_1,t_1)$ под $H_1$. На взгляде $k$:

- efficacy $b_k$ отсекает правый хвост: $\int_{b_k}^\infty p_k^{eff}\,dx=\Delta\alpha_k$;
- futility $a_k$ отсекает левую часть: $\int_{-\infty}^{a_k} p_k^{fut}\,dx=\Delta\beta_k$.

Между взглядами сворачиваем с приращением $B(t_{k+1})-B(t_k)\sim\mathcal N(\theta\Delta_{k+1},\Delta_{k+1})$: $p_{k+1}^{fut}(y)=\int_{a_k}^{b_k}p_k^{fut}(x)\varphi(y;x+\theta\Delta_{k+1},\Delta_{k+1})dx$. Возврат на шкалу $Z$: $c_k=b_k/\sqrt{t_k}$, $a_k=a_k/\sqrt{t_k}$.

</details>


In [11]:
def alpha_spending(t, alpha=0.05, family="obf"):
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return 2 - 2 * norm.cdf(norm.ppf(1 - alpha / 2) / np.sqrt(t))
    if family == "pocock":
        return alpha * np.log(1 + (np.e - 1) * t)
    raise ValueError(family)


def beta_spending(t, beta=0.20, family="obf"):
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return beta * t ** 3
    if family == "pocock":
        return beta * t
    raise ValueError(family)


def _convolve(p, grid, loc, scale):
    dx = grid[1] - grid[0]
    K = norm.pdf(grid[:, None], loc=grid[None, :] + loc, scale=scale)
    return (K @ p) * dx


def _quantile(p, grid, target, tail):
    dx = grid[1] - grid[0]
    cdf = np.cumsum(p) * dx
    vals = (cdf[-1] - cdf) if tail else cdf
    return grid[int(np.argmin(np.abs(vals - target)))]


def group_sequential_boundaries(t, alpha=0.05, beta=0.20, theta=None,
                                fam_alpha="obf", fam_beta="obf", grid_dx=0.01, n_sigma=6.0):
    t = np.asarray(t, dtype=float)
    K = len(t)
    if theta is None:
        theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    dt = np.diff(np.concatenate(([0.0], t)))
    grid = np.arange(-n_sigma, n_sigma + grid_dx, grid_dx)
    d_alpha = np.diff(np.concatenate(([0.0], alpha_spending(t, alpha, fam_alpha))))
    d_beta = np.diff(np.concatenate(([0.0], beta_spending(t, beta, fam_beta))))
    c = np.empty(K); a = np.empty(K)
    p_eff = norm.pdf(grid, loc=0.0, scale=math.sqrt(dt[0]))
    p_fut = norm.pdf(grid, loc=theta * dt[0], scale=math.sqrt(dt[0]))
    for k in range(K):
        b_k = _quantile(p_eff, grid, d_alpha[k], tail=True)
        c[k] = b_k / math.sqrt(t[k])
        a_k = _quantile(p_fut, grid, d_beta[k], tail=False)
        a[k] = a_k / math.sqrt(t[k])
        if k < K - 1:
            d = dt[k + 1]
            p_eff = _convolve(p_eff * (grid <= b_k), grid, 0.0, math.sqrt(d))
            p_fut = _convolve(p_fut * ((grid > a_k) & (grid <= b_k)), grid, theta * d, math.sqrt(d))
    return c, a


c_bound, a_bound = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde)
pd.DataFrame({
    "взгляд k": np.arange(1, design.K + 1),
    "t_k": design.t,
    "c_k (efficacy)": np.round(c_bound, 3),
    "a_k (futility)": np.round(a_bound, 3),
})


,взгляд k,t_k,c_k (efficacy),a_k (futility)
0,1,0.30,3.378,-1.205
1,2,0.50,2.531,-0.240
2,3,0.70,2.104,0.538
3,4,0.85,1.909,1.063
4,5,1.00,1.760,1.630


In [12]:
tt = np.asarray(design.t)
fig = go.Figure()
fig.add_trace(go.Scatter(x=tt, y=alpha_spending(tt, design.alpha, "obf"), mode="lines+markers",
                         line=dict(color=C_EFF, width=2.5), name="α-spending (OBF)"))
fig.add_trace(go.Scatter(x=tt, y=beta_spending(tt, design.beta, "obf"), mode="lines+markers",
                         line=dict(color=C_FUT, width=2.5), name="β-spending (ρ=3)"))
fig.add_hline(y=design.alpha, line_dash="dash", line_color=C_EFF, annotation_text="бюджет α")
fig.add_hline(y=design.beta, line_dash="dash", line_color=C_FUT, annotation_text="бюджет β")
fig = style(fig, title="Две стороны одной монеты: как «тратится» ошибка I и II рода",
            xlabel="доля информации t", ylabel="накопленная ошибка")
fig


In [13]:
tt = np.asarray(design.t)
bottom, top = -4.2, 4.2

fig = go.Figure()
# зона futility (ниже a_k) и зона efficacy (выше c_k)
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[a_bound, bottom, bottom],
                         fill="toself", fillcolor="rgba(217,95,2,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[c_bound, top, top],
                         fill="toself", fillcolor="rgba(27,158,119,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=tt, y=c_bound, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy c_k"))
fig.add_trace(go.Scatter(x=tt, y=a_bound, mode="lines+markers", line=dict(color=C_FUT, width=2.8), name="futility a_k"))
fig.add_annotation(x=0.66, y=3.4, text="СТОП — эффект", showarrow=False, font=dict(color=C_EFF, size=13))
fig.add_annotation(x=0.66, y=-2.4, text="СТОП — futility", showarrow=False, font=dict(color=C_FUT, size=13))
fig.add_annotation(x=0.66, y=0.9, text="продолжаем", showarrow=False, font=dict(size=13))
fig = style(fig, height=480, title="Двухграничный групповой-последовательный дизайн",
            xlabel="доля информации t", ylabel="Z(t)")
fig.update_yaxes(range=[bottom, top])
fig


## 5.1. Расширение на двустороннюю проверку

<details>
<summary>Развернуть</summary>

Для двустороннего теста: efficacy — $|Z_t|>c_k$ (α делится на два хвоста), futility — $|Z_t|<a_k$ (эффект «зажат» в коридоре вокруг нуля). Геометрически futility — «воронка» вокруг нуля, efficacy — два симметричных хвоста.

</details>


In [14]:
z_alpha2 = norm.ppf(1 - design.alpha / 2)
theta2 = z_alpha2 + norm.ppf(1 - design.beta)
c2, a2 = group_sequential_boundaries(design.t, design.alpha, design.beta, theta2)
tt = np.asarray(design.t)
bottom, top = -4.2, 4.2

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[a2, -a2[::-1], a2[0]],
                         fill="toself", fillcolor="rgba(217,95,2,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False, name="futility"))
fig.add_trace(go.Scatter(x=tt, y=c2, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="± efficacy"))
fig.add_trace(go.Scatter(x=tt, y=-c2, mode="lines+markers", line=dict(color=C_EFF, width=2.8), showlegend=False))
fig.add_trace(go.Scatter(x=tt, y=a2, mode="lines+markers", line=dict(color=C_FUT, width=2.8), name="± futility"))
fig.add_trace(go.Scatter(x=tt, y=-a2, mode="lines+markers", line=dict(color=C_FUT, width=2.8), showlegend=False))
fig = style(fig, height=480, title="Двусторонний дизайн: futility = коридор |Z| < a_k",
            xlabel="доля информации t", ylabel="Z(t)")
fig.update_yaxes(range=[bottom, top])
fig


## 6. Binding vs non-binding futility

<details>
<summary>Идея и доказательство (развернуть)</summary>

Futility-граница влияет на то, какие траектории доживают до конца. Вопрос: как учитывать её при расчёте α?

- **Binding**: futility-стоп обязателен, α считается с удалением траекторий ниже $a_k$. Если проигнорировать стоп — α вырастет.
- **Non-binding**: α считается так, *будто futility-границы нет*. Стоп можно отменить без риска для α.

**Почему binding α ≤ non-binding.** В non-binding область продолжения — «ещё не пересекли efficacy» ($-\infty<Z\le c_j$); в binding она дополнительно ограничена снизу ($a_j<Z\le c_j$). Продолжающееся множество в binding — подмножество, поэтому масса, достигающая efficacy, не больше.

</details>

График ниже показывает разницу: слева non-binding (зона продолжения — всё ниже efficacy-границы), справа binding (зона продолжения сужена снизу futility-границей). В обоих случаях считаем α efficacy-границы.


In [15]:
def overall_alpha_upper(c, t, lower=None, n_sim=400000):
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(0, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    active = np.ones(n_sim, dtype=bool)
    crossed = np.zeros(n_sim, dtype=bool)
    for k in range(len(t)):
        if lower is not None:
            active &= ~(Z[:, k] < lower[k]) & ~crossed
        crossed |= active & (Z[:, k] > c[k])
        active &= ~crossed
    return crossed.mean()


t = np.asarray(design.t)
alpha_nonb = overall_alpha_upper(c_bound, t, lower=None)
alpha_bind = overall_alpha_upper(c_bound, t, lower=a_bound)

tt = np.asarray(design.t)
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"Non-binding: α = {alpha_nonb:.4f}", f"Binding: α = {alpha_bind:.4f}"))

# non-binding: зона продолжения = всё ниже efficacy
fig.add_trace(go.Scatter(x=tt, y=c_bound, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy"), row=1, col=1)
fig.add_trace(go.Scatter(x=tt, y=a_bound, mode="lines+markers", line=dict(color=C_FUT, width=2.5, dash="dot"),
                         name="futility (игнорируется в α)"), row=1, col=1)

# binding: зона продолжения = между a_k и c_k
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[c_bound, a_bound[::-1], c_bound[0]],
                         fill="toself", fillcolor="rgba(153,153,153,0.15)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=tt, y=c_bound, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy", showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=tt, y=a_bound, mode="lines+markers", line=dict(color=C_FUT, width=2.8), name="futility", showlegend=False), row=1, col=2)

fig.update_layout(height=440, title="Разница binding и non-binding в зоне продолжения (и в итоговом α)",
                  template="plotly_white")
fig.update_xaxes(title_text="t", showgrid=True, gridcolor="rgba(0,0,0,0.07)")
fig.update_yaxes(title_text="Z(t)", showgrid=True, gridcolor="rgba(0,0,0,0.07)")
fig


## 7. Влияние futility на ошибки I/II рода + Monte-Carlo

<details>
<summary>Почему futility НЕ раздувает α, но снижает мощность (развернуть)</summary>

**Futility не раздувает α.** Ошибка I рода — траектория, которая под $H_0$ забралась выше efficacy. Futility отбрасывает траектории *ниже* себя; они и так редко достигали бы efficacy, а стоп лишь уменьшает их число. Вероятность отвергнуть при $H_0$ может только уменьшиться. Аргумент использует вложенность и монотонность, а не нормальность.

**Futility снижает мощность.** Под $H_1$ часть траекторий на раннем взгляде случайно выглядит слабо — futility зря их останавливает. Это плата за экономию трафика.

</details>

Симулируем групповой-последовательный дизайн и измеряем операционные характеристики. Ниже — результаты с человеческими названиями.


In [16]:
def simulate_trial(theta, t, c, a, n_sim=50000, futility="boundary", gamma=0.15,
                    theta_cp=None, theta0=0.0, tau=0.8, z_alpha=1.6449):
    """Симуляция группового-последовательного дизайна.

    futility: "none" | "boundary" (фиксированная a_k) | "cp" (CP<gamma) | "ppos" (PPoS<gamma).
    Исходы: efficacy / futility / complete; возвращает исход и долю информации на остановке.
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    outcome = np.full(n_sim, "complete")
    stop_frac = np.ones(n_sim)
    for k in range(len(t)):
        active = outcome == "complete"
        if not active.any():
            break
        eff = active & (Z[:, k] > c[k])
        still = active & ~eff
        if k == len(t) - 1:
            # финальный взгляд: futility = Z < a_K (фиксированная граница);
            # CP/PPoS тут вырождены (t=1 → sqrt(1-t)=0), поэтому их не считаем.
            fut = still & (Z[:, k] < a[k])
        elif futility == "boundary":
            fut = still & (Z[:, k] < a[k])
        elif futility == "cp":
            cp = conditional_power(Z[still, k], t[k], theta_cp, z_alpha)
            fut = still.copy(); fut[still] = cp < gamma
        elif futility == "ppos":
            pp = ppos(Z[still, k], t[k], theta0, tau, z_alpha)
            fut = still.copy(); fut[still] = pp < gamma
        else:
            fut = np.zeros_like(still)
        outcome[eff] = "efficacy"; outcome[fut] = "futility"
        stop_frac[eff] = t[k]; stop_frac[fut] = t[k]
    return outcome, stop_frac


def oc(theta, t, c, a, n_sim=50000, **kw):
    out, stop = simulate_trial(theta, t, c, a, n_sim, **kw)
    return {
        "FPR / power": (out == "efficacy").mean(),
        "P(futility stop)": (out == "futility").mean(),
        "P(дошли до конца)": (out == "complete").mean(),
        "E[N]/N_max": stop.mean(),
    }


In [17]:
t = np.asarray(design.t)
n_sim = 50000

rows = []
for label, theta in [("H0: эффекта нет (θ = 0)", 0.0), ("H1: эффект = MDE (θ = θ_MDE)", design.theta_mde)]:
    r = oc(theta, t, c_bound, a_bound, n_sim)
    r["Сценарий"] = label
    rows.append(r)

mc = pd.DataFrame(rows)[["Сценарий", "FPR / power", "P(futility stop)", "P(дошли до конца)", "E[N]/N_max"]]
mc = mc.rename(columns={"FPR / power": "Вероятность efficacy (под H0 — FPR, под H1 — мощность)"})
mc.round(4)


,Сценарий,"Вероятность efficacy (под H0 — FPR, под H1 — мощность)",P(futility stop),P(дошли до конца),E[N]/N_max
0,H0: эффекта нет (θ = 0),0.0483,0.9439,0.0078,0.6487
1,H1: эффект = MDE (θ = θ_MDE),0.7790,0.1981,0.0229,0.7418


In [18]:
# Операционные характеристики по спектру эффектов: три доли, в сумме = 1 (stacked area).
thetas = np.linspace(-0.2, 3.0, 17)
N = 30000
out_df = pd.DataFrame([oc(th, t, c_bound, a_bound, N) for th in thetas])
out_df["θ"] = thetas

fig = go.Figure()
fig.add_trace(go.Scatter(x=thetas, y=out_df["FPR / power"], mode="lines", stackgroup="one",
                         line=dict(color=C_EFF, width=0), fillcolor="rgba(27,158,119,0.6)", name="efficacy (power/FPR)"))
fig.add_trace(go.Scatter(x=thetas, y=out_df["P(futility stop)"], mode="lines", stackgroup="one",
                         line=dict(color=C_FUT, width=0), fillcolor="rgba(217,95,2,0.6)", name="futility stop"))
fig.add_trace(go.Scatter(x=thetas, y=out_df["P(дошли до конца)"], mode="lines", stackgroup="one",
                         line=dict(color=C_CONT, width=0), fillcolor="rgba(153,153,153,0.5)", name="дошли до конца"))
fig.add_vline(x=design.theta_mde, line_dash="dash", line_color="black", annotation_text="θ_MDE")
fig = style(fig, height=460,
            title="Операционные характеристики: три полосы в сумме дают 1 (при любом θ исход ровно один)",
            xlabel="истинный дрейф θ", ylabel="доля исходов")
fig


In [19]:
# Распределение исхода: столбцы под H0 и H1 (понятнее, чем сотни линий ансамбля).
n_sim = 50000
res0 = oc(0.0, t, c_bound, a_bound, n_sim)
res1 = oc(design.theta_mde, t, c_bound, a_bound, n_sim)

fig = go.Figure()
cats = ["efficacy", "futility", "дошли до конца"]
fig.add_trace(go.Bar(x=cats, y=[res0["FPR / power"], res0["P(futility stop)"], res0["P(дошли до конца)"]],
                     name="H0 (θ = 0)", marker_color=C_CONT))
fig.add_trace(go.Bar(x=cats, y=[res1["FPR / power"], res1["P(futility stop)"], res1["P(дошли до конца)"]],
                     name="H1 (θ = MDE)", marker_color=C_EFF))
fig = style(fig, height=420, title="Распределение исходов под H0 и H1",
            xlabel="исход", ylabel="вероятность")
fig.update_yaxes(range=[0, 1])
fig


## 8. Практические рекомендации

<details>
<summary>Развернуть</summary>

1. **Взгляды — не раньше 30–50% данных.** При малом $t$ карта CP почти плоская, ранний стоп зря теряет мощность.
2. **Порог CP/PPoS — 10–20%.** Выше — потеря мощности, ниже 5% — экономии почти нет.
3. **Выбор θ для CP** — как в разделе 2.
4. **Non-binding — дефолт:** стоп можно отменить, не испортив α.
5. **Полный дизайн = α-spending + β-spending.**

</details>


## 9. Сквозной пример: сравнение всех методов

Теперь соберём всё вместе на конкретном примере. Метрика — **средний чек** ($\sigma$ варьируемо, контроль $\mu_0$ не влияет на стандартизованный результат). Сравним, как разные правила остановки ведут себя по мощности, ошибке I рода и экономии трафика.

Два замечания к таблице: **(1)** binding и non-binding здесь используют одни и те же границы — они различаются только расчётом α (у binding он ниже), мощность и ASN у них совпадают; **(2)** PPoS-futility со **скептическим** prior ($\theta_0=0$, $\tau=0.8$) очень консервативен — видно, как сильно prior влияет на результат; с «дизайновым» prior ($\theta_0=\theta_{MDE}$) он близок к CP.


In [20]:
def compare_methods(sigma=100.0, mde=10.0, alpha=0.05, beta=0.20, K=5,
                     fam_alpha="obf", fam_beta="obf", gamma=0.15, theta0=0.0, tau=0.8, n_sim=20000):
    delta = mde / sigma
    theta_mde = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    z_alpha = norm.ppf(1 - alpha)
    t = np.linspace(0.3, 1.0, K)
    c, a = group_sequential_boundaries(t, alpha, beta, theta_mde, fam_alpha, fam_beta)

    methods = {}
    # 1) фиксированный z-тест (один взгляд в конце)
    methods["Фиксированный z-тест"] = dict(t=np.array([1.0]), c=np.array([z_alpha]), a=np.array([-np.inf]), futility="none")
    # 2) GST только efficacy
    methods["GST (efficacy only)"] = dict(t=t, c=c, a=np.full(K, -np.inf), futility="none")
    # 3) GST + futility non-binding
    methods["GST + futility (non-binding)"] = dict(t=t, c=c, a=a, futility="boundary")
    # 4) GST + futility binding (те же границы, но α считаем binding)
    methods["GST + futility (binding)"] = dict(t=t, c=c, a=a, futility="boundary", binding=True)
    # 5) GST + CP-futility
    methods["GST + CP-futility"] = dict(t=t, c=c, a=a, futility="cp", gamma=gamma, theta_cp=theta_mde)
    # 6) GST + PPoS-futility
    methods["GST + PPoS-futility"] = dict(t=t, c=c, a=a, futility="ppos", gamma=gamma, theta0=theta0, tau=tau)

    rows = []
    for name, m in methods.items():
        r0 = oc(0.0, m["t"], m["c"], m["a"], n_sim, futility=m["futility"],
                gamma=m.get("gamma", 0.15), theta_cp=m.get("theta_cp", theta_mde),
                theta0=m.get("theta0", 0.0), tau=m.get("tau", 0.8))
        r1 = oc(theta_mde, m["t"], m["c"], m["a"], n_sim, futility=m["futility"],
                gamma=m.get("gamma", 0.15), theta_cp=m.get("theta_cp", theta_mde),
                theta0=m.get("theta0", 0.0), tau=m.get("tau", 0.8))
        fpr = r0["FPR / power"]
        if m.get("binding"):
            fpr = overall_alpha_upper(m["c"], m["t"], lower=m["a"], n_sim=200000)
        rows.append({
            "Метод": name,
            "FPR (H0)": fpr,
            "Мощность (H1)": r1["FPR / power"],
            "P(futility | H1)": r1["P(futility stop)"],
            "E[N]/N_max (H1)": r1["E[N]/N_max"],
        })
    df = pd.DataFrame(rows)
    return df, theta_mde


cmp_df, theta_mde = compare_methods()
cmp_df.round(4)


,Метод,FPR (H0),Мощность (H1),P(futility | H1),E[N]/N_max (H1)
0,Фиксированный z-тест,0.0518,0.8000,0.0000,1.0000
1,GST (efficacy only),0.0503,0.7836,0.0000,0.7750
2,GST + futility (non-binding),0.0474,0.7802,0.1960,0.7417
3,GST + futility (binding),0.0488,0.7752,0.2004,0.7432
4,GST + CP-futility,0.0495,0.7805,0.1919,0.7530
5,GST + PPoS-futility,0.0298,0.5855,0.4060,0.5558


In [21]:
fig = go.Figure()
fig.add_trace(go.Bar(x=cmp_df["Метод"], y=cmp_df["Мощность (H1)"], name="Мощность (H1)", marker_color=C_EFF))
fig.add_trace(go.Bar(x=cmp_df["Метод"], y=cmp_df["E[N]/N_max (H1)"], name="E[N]/N_max (экономия)", marker_color=C_CONT))
fig.add_trace(go.Bar(x=cmp_df["Метод"], y=cmp_df["FPR (H0)"], name="FPR (H0)", marker_color=C_FUT, visible="legendonly"))
fig = style(fig, height=480, title="Сравнение методов: мощность, экономия трафика и ошибка I рода",
            xlabel="метод", ylabel="значение")
fig.update_xaxes(tickangle=-20)
fig


In [22]:
def plot_comparison(K=5, sigma=100.0, mde=10.0, fam_alpha="obf", fam_beta="obf", gamma=0.15):
    df, _ = compare_methods(sigma=sigma, mde=mde, K=K, fam_alpha=fam_alpha, fam_beta=fam_beta, gamma=gamma, n_sim=8000)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df["Метод"], y=df["Мощность (H1)"], name="Мощность (H1)", marker_color=C_EFF))
    fig.add_trace(go.Bar(x=df["Метод"], y=df["E[N]/N_max (H1)"], name="E[N]/N_max", marker_color=C_CONT))
    fig.add_trace(go.Bar(x=df["Метод"], y=df["FPR (H0)"], name="FPR (H0)", marker_color=C_FUT, visible="legendonly"))
    fig = style(fig, height=460, title=f"Сравнение методов (K={K}, σ={sigma}, MDE={mde})",
                xlabel="метод", ylabel="значение")
    fig.update_xaxes(tickangle=-20)
    return go.FigureWidget(fig)

interact(plot_comparison,
         K=IntSlider(value=5, min=1, max=50, step=1, description="взглядов K"),
         sigma=FloatSlider(value=100.0, min=10, max=500, step=10, description="σ метрики"),
         mde=FloatSlider(value=10.0, min=1, max=50, step=1, description="MDE"),
         fam_alpha=Dropdown(options=["obf", "pocock"], value="obf", description="α-spending"),
         fam_beta=Dropdown(options=["obf", "pocock"], value="obf", description="β-spending"),
         gamma=FloatSlider(value=0.15, min=0.05, max=0.40, step=0.05, description="порог CP"))


interactive(children=(IntSlider(value=5, description='взглядов K', max=50, min=1), FloatSlider(value=100.0, de…

<function __main__.plot_comparison(K=5, sigma=100.0, mde=10.0, fam_alpha='obf', fam_beta='obf', gamma=0.15)>

## 10. Ненормальность метрики

<details>
<summary>Развернуть</summary>

Математика опирается на нормальность **статистики**, а не метрики (ЦПТ).

- **Независимость приращений** — точное свойство *дизайна*, от распределения не зависит; ломается только кластерами/carryover.
- **Нормальность $Z_t$** — *асимптотика*: при «приятной» метрике и достаточном $n$ границы валидны с точностью $O(1/\sqrt n)$; при тяжёлых хвостах и малом $n$ — смещение (лечится студентизацией, information-based monitoring, симуляционным дизайном).

Качественные факты («futility не раздувает α», логика β-spending, non-binding) от нормальности не зависят.

</details>


## 11. Итоги

1. **Futility не раздувает ошибку I рода** — в отличие от peeking; она лишь снижает мощность.
2. **CP** — шанс дойти до значимости при фиксированном θ; **PPoS** усредняет по неопределённости θ; CP — частный случай PPoS.
3. **Stochastic curtailment** — общая рамка CP и PPoS.
4. **β-spending — зеркало α-spending**: априорная futility-граница с контролем ошибки II рода.
5. **Non-binding** — безопасный дефолт.

| | CP | PPoS | β-spending |
|---|---|---|---|
| Философия | частотная | байесовская | частотная (дизайн) |
| Эффект | фиксированный θ | prior на θ | θ_MDE |
| Граница | данные-зависимая | данные-зависимая | фиксирована заранее |
| Контролирует | правило «стоп» | правило «стоп» | ошибку II рода |
| Когда | быстро оценить шансы | учесть неопределённость | спроектировать тест |
